# Calculus for Automatic Differentiation

A pen-and-paper study guide. This notebook builds from basic derivative rules
through multivariate calculus to reverse-mode autodiff on computational graphs.

Each section ends with exercises. Work them out on paper before checking.

In [ ]:
%pip install graphviz

In [ ]:
import graphviz

def make_graph(edges, node_labels=None, highlights=None, edge_labels=None):
    """
    Helper to render a computation graph.
    
    edges:       list of (parent, child) tuples — arrow points from input to output
    node_labels: dict mapping node id -> display label
    highlights:  set of node ids to highlight in blue
    edge_labels: dict mapping (parent, child) -> label string
    """
    g = graphviz.Digraph(graph_attr={'rankdir': 'BT', 'size': '6,6'},
                         node_attr={'shape': 'record', 'fontname': 'Courier'})
    node_labels = node_labels or {}
    highlights = highlights or set()
    edge_labels = edge_labels or {}
    
    nodes = set()
    for a, b in edges:
        nodes.add(a)
        nodes.add(b)
    
    for n in nodes:
        label = node_labels.get(n, n)
        style = 'filled' if n in highlights else ''
        fillcolor = 'lightblue' if n in highlights else 'white'
        g.node(n, label=label, style=style, fillcolor=fillcolor)
    
    for a, b in edges:
        elabel = edge_labels.get((a, b), '')
        g.edge(a, b, label=elabel)
    
    return g

---
## 1. The Derivative as a Symbolic Rule

You already know the limit definition:

$$f'(x) = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$$

The key insight from Leibniz is that we don't need to evaluate this limit every time.
Instead, we can derive **rules** that map each type of expression to its derivative.
These rules are the foundation of autodiff — a computer can apply them mechanically.

### The rules you need

| Function $f(x)$ | Derivative $f'(x)$ | Name |
|---|---|---|
| $c$ (constant) | $0$ | Constant rule |
| $x$ | $1$ | Identity rule |
| $x^n$ | $n \cdot x^{n-1}$ | Power rule |
| $e^x$ | $e^x$ | Exponential rule |
| $\ln(x)$ | $1/x$ | Log rule |

And for combining expressions:

| Expression | Derivative | Name |
|---|---|---|
| $f(x) + g(x)$ | $f'(x) + g'(x)$ | Sum rule |
| $f(x) - g(x)$ | $f'(x) - g'(x)$ | Difference rule |
| $f(x) \cdot g(x)$ | $f'(x) \cdot g(x) + f(x) \cdot g'(x)$ | Product rule |
| $f(x) / g(x)$ | $\frac{f'(x) \cdot g(x) - f(x) \cdot g'(x)}{g(x)^2}$ | Quotient rule |

### Exercises (pen and paper)

Differentiate each with respect to $x$. Apply the rules mechanically.

1. $f(x) = 5x^3 + 2x$
2. $f(x) = x^2 \cdot e^x$
3. $f(x) = \frac{x^2}{x + 1}$
4. $f(x) = x \cdot \ln(x) - x$

<details><summary><b>Solutions</b></summary>

1. $f'(x) = 15x^2 + 2$

2. Product rule: $f'(x) = 2x \cdot e^x + x^2 \cdot e^x = e^x(2x + x^2)$

3. Quotient rule: $f'(x) = \frac{2x(x+1) - x^2 \cdot 1}{(x+1)^2} = \frac{x^2 + 2x}{(x+1)^2}$

4. Product rule on first term: $f'(x) = 1 \cdot \ln(x) + x \cdot \frac{1}{x} - 1 = \ln(x) + 1 - 1 = \ln(x)$

</details>

---
## 2. The Chain Rule

The chain rule handles **function composition**: applying one function to the result of another.

If $y = f(g(x))$, then:

$$\frac{dy}{dx} = \frac{df}{dg} \cdot \frac{dg}{dx}$$

In words: **the derivative of the outer, times the derivative of the inner.**

This is the single most important rule for autodiff. Every operation in a
computation graph is a function applied to the result of previous operations.
The chain rule lets us differentiate through the entire chain.

### Example

$y = e^{x^2}$

Let $g(x) = x^2$ (inner) and $f(g) = e^g$ (outer). Then:

$$\frac{dy}{dx} = \underbrace{e^{g}}_{\text{outer derivative}} \cdot \underbrace{2x}_{\text{inner derivative}} = 2x \cdot e^{x^2}$$

### Longer chains

The chain rule extends to any length. If $y = f(g(h(x)))$:

$$\frac{dy}{dx} = \frac{df}{dg} \cdot \frac{dg}{dh} \cdot \frac{dh}{dx}$$

Each link in the chain contributes one factor.

### Exercises

1. $y = \ln(x^2 + 1)$
2. $y = (3x + 5)^4$
3. $y = e^{\ln(x)}$ (simplify first, then verify with chain rule)
4. $y = \ln(e^x)$ (same idea)

<details><summary><b>Solutions</b></summary>

1. Outer: $\ln(u) \to 1/u$, inner: $u = x^2+1 \to 2x$. Result: $\frac{2x}{x^2+1}$

2. Outer: $u^4 \to 4u^3$, inner: $u = 3x+5 \to 3$. Result: $12(3x+5)^3$

3. $e^{\ln(x)} = x$, so $dy/dx = 1$. Chain rule: $e^{\ln(x)} \cdot \frac{1}{x} = \frac{x}{x} = 1$ ✓

4. $\ln(e^x) = x$, so $dy/dx = 1$. Chain rule: $\frac{1}{e^x} \cdot e^x = 1$ ✓

</details>

---
## 3. Partial Derivatives

So far, every function had **one** input $x$. Now consider a function of
**two** inputs:

$$f(x, y) = x^2 \cdot y + y^3$$

The **partial derivative** $\frac{\partial f}{\partial x}$ answers:
"if I nudge $x$ slightly while holding $y$ fixed, how much does $f$ change?"

The rule is simple: **treat every other variable as a constant**, then
differentiate normally.

$$\frac{\partial f}{\partial x} = 2xy \qquad \frac{\partial f}{\partial y} = x^2 + 3y^2$$

In the first case, $y$ is a constant, so $x^2 \cdot y$ differentiates like
$c \cdot x^2 \to 2cx = 2xy$, and $y^3$ is a constant so its derivative is 0.

### Notation

- $\frac{\partial f}{\partial x}$ — partial derivative of $f$ with respect to $x$
- $\frac{df}{dx}$ — total derivative (used when $f$ depends on $x$ only, or through intermediaries)
- $\nabla f = \left(\frac{\partial f}{\partial x}, \frac{\partial f}{\partial y}\right)$ — the **gradient**, a vector of all partial derivatives

The gradient tells you the direction of steepest increase of $f$.

### Exercises

Compute all partial derivatives.

1. $f(x, y) = x^3 + xy + y^2$
2. $f(x, y) = e^{xy}$  (hint: chain rule, treating $xy$ as the inner function)
3. $f(x, y, z) = x \cdot y \cdot z$
4. $f(a, b) = \frac{a}{b}$

<details><summary><b>Solutions</b></summary>

1. $\frac{\partial f}{\partial x} = 3x^2 + y$, $\quad \frac{\partial f}{\partial y} = x + 2y$

2. $\frac{\partial f}{\partial x} = y \cdot e^{xy}$, $\quad \frac{\partial f}{\partial y} = x \cdot e^{xy}$

3. $\frac{\partial f}{\partial x} = yz$, $\quad \frac{\partial f}{\partial y} = xz$, $\quad \frac{\partial f}{\partial z} = xy$

4. $\frac{\partial f}{\partial a} = \frac{1}{b}$, $\quad \frac{\partial f}{\partial b} = -\frac{a}{b^2}$

</details>

---
## 4. The Multivariate Chain Rule

This is where the confusion you described comes in. Here's the scenario:

$$L = f(u, v) \quad \text{where} \quad u = g(\theta) \quad \text{and} \quad v = h(\theta)$$

$L$ depends on $\theta$ through **two different paths**: one via $u$, one via $v$.
The total effect of nudging $\theta$ is the **sum** of its effects along each path:

$$\frac{dL}{d\theta} = \frac{\partial L}{\partial u} \cdot \frac{du}{d\theta} + \frac{\partial L}{\partial v} \cdot \frac{dv}{d\theta}$$

This is the **multivariate chain rule**: sum over all paths from output to input.

### Why summation?

Think physically. If you nudge $\theta$ by a tiny amount $\epsilon$:
- $u$ changes by $\frac{du}{d\theta} \cdot \epsilon$, which changes $L$ by $\frac{\partial L}{\partial u} \cdot \frac{du}{d\theta} \cdot \epsilon$
- $v$ changes by $\frac{dv}{d\theta} \cdot \epsilon$, which changes $L$ by $\frac{\partial L}{\partial v} \cdot \frac{dv}{d\theta} \cdot \epsilon$

These effects happen **simultaneously and independently**, so they add up.
This is true regardless of what $f$ is — it could be addition, multiplication,
division, anything. The summation comes from the chain rule, not from $f$.

In [ ]:
# Visualize: θ feeds into L through two paths
make_graph(
    edges=[('θ', 'u'), ('θ', 'v'), ('u', 'L'), ('v', 'L')],
    node_labels={'θ': 'θ', 'u': 'u = g(θ)', 'v': 'v = h(θ)', 'L': 'L = f(u, v)'},
    edge_labels={
        ('θ', 'u'): ' du/dθ',
        ('θ', 'v'): ' dv/dθ',
        ('u', 'L'): ' ∂L/∂u',
        ('v', 'L'): ' ∂L/∂v',
    },
    highlights={'θ'},
)

### Concrete example: $\theta / \theta$

Let $L = u / v$ where $u = \theta$ and $v = \theta$.

- Path through $u$: $\frac{\partial L}{\partial u} = \frac{1}{v} = \frac{1}{\theta}$, $\quad \frac{du}{d\theta} = 1$
- Path through $v$: $\frac{\partial L}{\partial v} = -\frac{u}{v^2} = -\frac{\theta}{\theta^2} = -\frac{1}{\theta}$, $\quad \frac{dv}{d\theta} = 1$

Total: $\frac{dL}{d\theta} = \frac{1}{\theta} \cdot 1 + \left(-\frac{1}{\theta}\right) \cdot 1 = 0$ ✓

The partial derivative $\frac{\partial L}{\partial u}$ treats $v$ as a **separate, independent** input.
It doesn't "know" that $u$ and $v$ are actually the same variable $\theta$. That fact
is accounted for by the multivariate chain rule, which sums the contributions
from both paths.

This is exactly how the `grad_map` accumulation works in `autodiff.py`:
division's backward rule sends $+1/\theta$ and $-1/\theta$ as separate contributions
to the same tensor, and the accumulator sums them to zero.

### Exercises

1. Let $L = u + v$ where $u = \theta^2$ and $v = \theta^3$. Compute $dL/d\theta$ via the multivariate chain rule. Verify by substituting first: $L = \theta^2 + \theta^3$.

2. Let $L = u \cdot v$ where $u = \theta$ and $v = \theta$. Compute $dL/d\theta$ via the chain rule. Verify: $L = \theta^2$, so $dL/d\theta = 2\theta$.

3. Let $L = u \cdot v$ where $u = e^\theta$ and $v = \ln(\theta)$. Compute $dL/d\theta$ via the chain rule.

4. Let $L = u - v$ where $u = \theta$ and $v = \theta$. What is $dL/d\theta$?

<details><summary><b>Solutions</b></summary>

1. $\frac{dL}{d\theta} = 1 \cdot 2\theta + 1 \cdot 3\theta^2 = 2\theta + 3\theta^2$ ✓

2. $\frac{\partial L}{\partial u} = v = \theta$, $\frac{\partial L}{\partial v} = u = \theta$, both paths contribute $\theta \cdot 1 = \theta$. Total: $\theta + \theta = 2\theta$ ✓

3. Path 1: $\frac{\partial L}{\partial u} \cdot \frac{du}{d\theta} = \ln(\theta) \cdot e^\theta$. Path 2: $\frac{\partial L}{\partial v} \cdot \frac{dv}{d\theta} = e^\theta \cdot \frac{1}{\theta}$. Total: $e^\theta \ln(\theta) + \frac{e^\theta}{\theta}$

4. $\frac{\partial L}{\partial u} = 1$, $\frac{\partial L}{\partial v} = -1$. Total: $1 \cdot 1 + (-1) \cdot 1 = 0$ ✓ (same as $\theta - \theta = 0$)

</details>

---
## 5. Computation Graphs

An expression like $L = (x + y) \cdot y$ can be drawn as a directed acyclic graph (DAG)
where each node is an operation and edges connect inputs to outputs.

Let's name the intermediate: $a = x + y$, then $L = a \cdot y$.

In [ ]:
make_graph(
    edges=[('x', 'a'), ('y', 'a'), ('a', 'L'), ('y', 'L')],
    node_labels={
        'x': 'x',
        'y': 'y',
        'a': 'a = x + y  |  +',
        'L': 'L = a * y  |  *',
    },
)

Notice that $y$ has **two outgoing edges** — it's used by both the `+` and the `*`.
When we differentiate $L$ w.r.t. $y$, we must sum the contributions from both paths.
This is the multivariate chain rule in action on a graph.

Every node's **local derivatives** are just the partial derivatives of that
operation with respect to each of its inputs:

| Node | Operation | $\partial/\partial$ left input | $\partial/\partial$ right input |
|---|---|---|---|
| $a$ | $x + y$ | $1$ | $1$ |
| $L$ | $a \cdot y$ | $y$ | $a$ |

These local derivatives are the building blocks. The chain rule tells us
how to combine them.

---
## 6. Reverse-Mode Autodiff (Backpropagation)

Now we put it all together. The algorithm:

1. **Seed**: Set $\overline{L} = \frac{\partial L}{\partial L} = 1$
2. **Walk backward** through the graph (from output to inputs)
3. At each node, multiply the incoming gradient by the local derivative and send it to each input
4. If an input receives gradients from **multiple** outputs, **sum** them

The notation $\overline{v}$ means $\frac{\partial L}{\partial v}$ — "the gradient of $L$ w.r.t. node $v$".

### Worked example

Using the graph from above: $a = x + y$, $L = a \cdot y$.

**Step 1**: Seed $\overline{L} = 1$

**Step 2**: Process node $L = a \cdot y$ (multiply):
- $\overline{L}$ flows to $a$: $\quad \overline{a} \mathrel{+}= \overline{L} \cdot \frac{\partial L}{\partial a} = 1 \cdot y = y$
- $\overline{L}$ flows to $y$: $\quad \overline{y} \mathrel{+}= \overline{L} \cdot \frac{\partial L}{\partial y} = 1 \cdot a = a = x + y$

**Step 3**: Process node $a = x + y$ (add):
- $\overline{a}$ flows to $x$: $\quad \overline{x} \mathrel{+}= \overline{a} \cdot 1 = y$
- $\overline{a}$ flows to $y$: $\quad \overline{y} \mathrel{+}= \overline{a} \cdot 1 = y$

**Result**:
- $\overline{x} = y$
- $\overline{y} = (x + y) + y = x + 2y$

**Verification**: $L = (x+y) \cdot y = xy + y^2$, so $\frac{\partial L}{\partial x} = y$ and $\frac{\partial L}{\partial y} = x + 2y$ ✓

Note how $y$ received gradient contributions from **two** nodes ($L$ and $a$),
and they were summed. This is the `+=` in the algorithm, and the
`grad_map[operand] = existing_grad + operand_grad` line in `autodiff.py`.

In [ ]:
# The same graph, now annotated with gradients flowing backward (shown on edges)
make_graph(
    edges=[('x', 'a'), ('y', 'a'), ('a', 'L'), ('y', 'L')],
    node_labels={
        'x': '{x | x̄ = y}',
        'y': '{y | ȳ = (x+y) + y}',
        'a': '{a = x + y | ā = y}',
        'L': '{L = a·y | L̄ = 1}',
    },
    edge_labels={
        ('a', 'L'): '  ∂L/∂a = y',
        ('y', 'L'): '  ∂L/∂y = a  ',
        ('x', 'a'): '  ∂a/∂x = 1',
        ('y', 'a'): '  ∂a/∂y = 1  ',
    },
    highlights={'x', 'y'},
)

---
## 7. Exercises: Trace the Algorithm on Paper

For each graph below, run the reverse-mode algorithm by hand:
1. Seed $\overline{\text{output}} = 1$
2. Process nodes from output to inputs
3. At each node, send gradient × local derivative to each input
4. Sum contributions when a node receives from multiple sources

### Exercise A

$a = x \cdot x$, $\quad L = a + x$

Find $\overline{x}$. Verify: $L = x^2 + x$, so $dL/dx = 2x + 1$.

In [ ]:
make_graph(
    edges=[('x', 'a'), ('x2', 'a'), ('a', 'L'), ('x3', 'L')],
    node_labels={
        'x': 'x', 'x2': 'x', 'x3': 'x',
        'a': 'a = x * x  |  *',
        'L': 'L = a + x  |  +',
    },
    highlights={'x', 'x2', 'x3'},
)

**Important**: All three "x" boxes are the **same variable**. In a real computation
graph (like resin's), they're the same tensor object. The graph above just draws
them separately so you can see the edges. When accumulating gradients, all
contributions to any "x" node are summed into one $\overline{x}$.

Here's the same expression as a proper DAG with one $x$ node:

In [ ]:
make_graph(
    edges=[('x', 'a'), ('a', 'L'), ('x', 'L')],
    node_labels={
        'x': 'x  (used 3 times)',
        'a': 'a = x * x  |  *',
        'L': 'L = a + x  |  +',
    },
    highlights={'x'},
)

Note: $x$ feeds into the `*` node **twice** (as both operands) and into the `+`
node once — three contributions total. The arrow from `x` to `a` accounts for
both operand slots of the `*` node.

<details><summary><b>Solution</b></summary>

Process $L = a + x$: $\quad \overline{a} \mathrel{+}= 1$, $\quad \overline{x} \mathrel{+}= 1$

Process $a = x \cdot x$: left operand is $x$, right operand is $x$.
- $\overline{x} \mathrel{+}= \overline{a} \cdot x = 1 \cdot x = x$ (from the right-operand slot)
- $\overline{x} \mathrel{+}= \overline{a} \cdot x = 1 \cdot x = x$ (from the left-operand slot)

Total: $\overline{x} = 1 + x + x = 2x + 1$ ✓

</details>

### Exercise B

$a = x + y$, $\quad b = a \cdot y$, $\quad L = b - x$

Find $\overline{x}$ and $\overline{y}$. Verify: $L = (x+y) \cdot y - x = xy + y^2 - x$.

In [ ]:
make_graph(
    edges=[('x', 'a'), ('y', 'a'), ('a', 'b'), ('y', 'b'), ('b', 'L'), ('x', 'L')],
    node_labels={
        'x': 'x',
        'y': 'y',
        'a': 'a = x + y  |  +',
        'b': 'b = a * y  |  *',
        'L': 'L = b - x  |  −',
    },
    highlights={'x', 'y'},
)

<details><summary><b>Solution</b></summary>

Process $L = b - x$: $\quad \overline{b} \mathrel{+}= 1$, $\quad \overline{x} \mathrel{+}= -1$

Process $b = a \cdot y$: $\quad \overline{a} \mathrel{+}= \overline{b} \cdot y = y$, $\quad \overline{y} \mathrel{+}= \overline{b} \cdot a = (x+y)$

Process $a = x + y$: $\quad \overline{x} \mathrel{+}= \overline{a} \cdot 1 = y$, $\quad \overline{y} \mathrel{+}= \overline{a} \cdot 1 = y$

Totals:
- $\overline{x} = -1 + y = y - 1$ ✓ (from $xy + y^2 - x$: $\partial/\partial x = y - 1$)
- $\overline{y} = (x + y) + y = x + 2y$ ✓ (from $xy + y^2 - x$: $\partial/\partial y = x + 2y$)

</details>

### Exercise C

$a = x^2$, $\quad b = \ln(a)$, $\quad L = b \cdot x$

Find $\overline{x}$. Verify: $L = x \cdot \ln(x^2) = 2x \cdot \ln(x)$.

In [ ]:
make_graph(
    edges=[('x', 'a'), ('a', 'b'), ('b', 'L'), ('x', 'L')],
    node_labels={
        'x': 'x',
        'a': 'a = x²  |  pow',
        'b': 'b = ln(a)  |  log',
        'L': 'L = b·x  |  *',
    },
    highlights={'x'},
)

<details><summary><b>Solution</b></summary>

Process $L = b \cdot x$: $\quad \overline{b} \mathrel{+}= x$, $\quad \overline{x} \mathrel{+}= b = \ln(x^2)$

Process $b = \ln(a)$: $\quad \overline{a} \mathrel{+}= \overline{b} \cdot \frac{1}{a} = \frac{x}{x^2} = \frac{1}{x}$

Process $a = x^2$: $\quad \overline{x} \mathrel{+}= \overline{a} \cdot 2x = \frac{1}{x} \cdot 2x = 2$

Total: $\overline{x} = \ln(x^2) + 2 = 2\ln(x) + 2$

Verify: $L = 2x\ln(x)$, so $\frac{dL}{dx} = 2\ln(x) + 2x \cdot \frac{1}{x} = 2\ln(x) + 2$ ✓

</details>

### Exercise D (final boss)

This one matches a realistic autodiff scenario. Given:

$a = x \cdot w$, $\quad b = a + bias$, $\quad c = e^b$, $\quad L = c \cdot c$

Find $\overline{x}$, $\overline{w}$, and $\overline{bias}$.

In [ ]:
make_graph(
    edges=[
        ('x', 'a'), ('w', 'a'),
        ('a', 'b'), ('bias', 'b'),
        ('b', 'c'),
        ('c', 'L'),
    ],
    node_labels={
        'x': 'x', 'w': 'w', 'bias': 'bias',
        'a': 'a = x·w  |  *',
        'b': 'b = a + bias  |  +',
        'c': 'c = exp(b)  |  exp',
        'L': 'L = c·c  |  *',
    },
    highlights={'x', 'w', 'bias'},
)

<details><summary><b>Solution</b></summary>

Process $L = c \cdot c$: $\quad \overline{c} \mathrel{+}= 1 \cdot c = c$ (left), $\quad \overline{c} \mathrel{+}= 1 \cdot c = c$ (right). Total: $\overline{c} = 2c$

Process $c = e^b$: $\quad \overline{b} \mathrel{+}= \overline{c} \cdot e^b = 2c \cdot c = 2c^2$ (since $e^b = c$)

Process $b = a + bias$: $\quad \overline{a} \mathrel{+}= 2c^2$, $\quad \overline{bias} \mathrel{+}= 2c^2$

Process $a = x \cdot w$: $\quad \overline{x} \mathrel{+}= \overline{a} \cdot w = 2c^2 w$, $\quad \overline{w} \mathrel{+}= \overline{a} \cdot x = 2c^2 x$

Results (substituting $c = e^{xw + bias}$):
- $\overline{x} = 2w \cdot e^{2(xw + bias)}$
- $\overline{w} = 2x \cdot e^{2(xw + bias)}$
- $\overline{bias} = 2e^{2(xw + bias)}$

Notice how $\overline{c} = 2c$ uses the **forward activation** $c$. This is why
`autodiff.py` passes `node` into the backward rules — to reuse the forward
values instead of recomputing them.

</details>

---
## 8. Summary: Mapping Calculus to Code

| Calculus concept | In `autodiff.py` |
|---|---|
| $\overline{v} = \frac{\partial L}{\partial v}$ | `grad_map[v]` |
| Seed: $\overline{L} = 1$ | `grad_map = {output: output.ones_like()}` |
| Local derivative $\frac{\partial(\text{op})}{\partial(\text{input})}$ | `_backward_elementwise()` rules |
| Chain rule: multiply by local derivative | `grad_output * (local deriv)` |
| Multivariate chain rule: sum over paths | `grad_map[operand] = existing_grad + operand_grad` |
| Walk output → inputs | `for node in topological_sort():` |
| Reuse forward activations | `node` parameter in backward rules (e.g., exp reuses $e^a$) |